<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-06-function-calling/lesson-6.2-calling-loop/notebooks/GCP_Capstone_6.2_CallingLoop.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6.2 Complete Calling Loop — LLM Decides → Code Executes → Result Returns → LLM Synthesizes
**Netsetos GenAI Engineering — GCP Capstone**

Build the production while-loop dispatcher, chain sequential calls, execute parallel calls, handle errors as data.


## Setup


In [ ]:
!pip install -q google-genai

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google import genai
from google.genai import types

client = genai.Client(enterprise=True, project=PROJECT_ID,
                      location='global')
print(f'Client ready for {PROJECT_ID}')


## Cell 1: Define DocuMind Functions


In [ ]:
def search_documents(query: str, doc_type: str = 'all', top_k: int = 5) -> dict:
    """Search DocuMind document collection by query.
    Args:
        query: Search query in natural language
        doc_type: Filter by type (research_paper, invoice, legal, form, all)
        top_k: Number of results to return
    """
    mock = {
        'legal': [{'id': 'D-01', 'title': 'NDA Template', 'pages': 8},
                  {'id': 'D-02', 'title': 'Service Agreement', 'pages': 24},
                  {'id': 'D-03', 'title': 'Employment Contract', 'pages': 15}],
        'invoice': [{'id': 'D-10', 'title': 'Q1 Invoice', 'pages': 2}],
    }
    if doc_type == 'all':
        results = [d for docs in mock.values() for d in docs]
    else:
        results = mock.get(doc_type, [])
    return {'documents': results[:top_k], 'total': len(results)}

def calculate_processing_cost(num_documents: int, total_pages: int,
                              processing_type: str = 'standard') -> dict:
    """Estimate document processing cost in USD and INR.
    Args:
        num_documents: Number of documents to process
        total_pages: Total page count across all documents
        processing_type: Tier — standard, priority, or bulk
    """
    rates = {'standard': 0.05, 'priority': 0.12, 'bulk': 0.03}
    cost = total_pages * rates.get(processing_type, 0.05)
    return {'num_documents': num_documents, 'total_pages': total_pages,
            'rate_per_page': rates.get(processing_type, 0.05),
            'cost_usd': round(cost, 2), 'cost_inr': round(cost * 85, 2)}

def get_usage_stats(metric: str, days: int = 7) -> dict:
    """Get DocuMind RAG pipeline usage statistics.
    Args:
        metric: Which metric (queries, costs, latency, users)
        days: Number of days to look back
    """
    data = {'queries': 1247, 'costs': 18.50, 'latency': 245, 'users': 42}
    return {'metric': metric, 'period': f'last {days} days',
            'value': data.get(metric, 0), 'trend': '+12%'}

FUNCTIONS = {
    'search_documents': search_documents,
    'calculate_processing_cost': calculate_processing_cost,
    'get_usage_stats': get_usage_stats,
}
print(f'Registered {len(FUNCTIONS)} functions')


## Cell 2: The While-Loop Dispatcher


In [ ]:
def run_function_loop(client, prompt, tools, functions,
                      model='gemini-3.6-flash',
                      system_instruction='', max_turns=10):
    """Production while-loop: call until model returns text."""
    # Disable Automatic Function Calling: WE dispatch each call here.
    # With raw callables in tools and AFC on (the default), the SDK
    # runs the whole loop itself and response.function_calls is empty.
    config = types.GenerateContentConfig(
        tools=tools, system_instruction=system_instruction,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True))
    contents = [types.Content(role='user', parts=[
        types.Part.from_text(text=prompt)])]

    for turn in range(max_turns):
        response = client.models.generate_content(
            model=model, contents=contents, config=config)
        contents.append(response.candidates[0].content)

        if not response.function_calls:
            return response.text  # Done!

        result_parts = []
        for fc in response.function_calls:
            print(f'  [Turn {turn+1}] {fc.name}({dict(fc.args)})')
            try:
                result = functions[fc.name](**fc.args)
                result_parts.append(
                    types.Part.from_function_response(
                        name=fc.name, response={'result': result}))
            except Exception as e:
                result_parts.append(
                    types.Part.from_function_response(
                        name=fc.name, response={'error': str(e)}))

        contents.append(types.Content(role='user', parts=result_parts))

    return 'Max turns reached.'

print('While-loop dispatcher ready')

## Cell 3: Single-Call Test


In [ ]:
# Simple single-call test
TOOLS = [search_documents, calculate_processing_cost, get_usage_stats]

answer = run_function_loop(
    client=client,
    prompt='How many queries did we get this week?',
    tools=TOOLS,
    functions=FUNCTIONS)
print('\n=== Answer ===')
print(answer)


## Cell 4: Sequential Chain Test


In [ ]:
# Sequential: search → then calculate cost based on results
answer = run_function_loop(
    client=client,
    prompt='Find all legal documents and estimate bulk processing cost',
    tools=TOOLS,
    functions=FUNCTIONS,
    system_instruction='You are DocuMind AI. When estimating costs, first '
                       'search for documents to get accurate page counts. '
                       'Never guess page numbers. If search returns no '
                       'results, explain this clearly.')
print('\n=== Sequential Chain Answer ===')
print(answer)


## Cell 5: Automatic Chat Session


In [ ]:
# Chat session with auto function calling
chat = client.chats.create(
    model='gemini-3.6-flash',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        system_instruction='You are DocuMind AI. Use tools to answer '
                           'document questions accurately.'))

# Turn 1: search
r1 = chat.send_message('What legal documents do we have?')
print(f'Turn 1: {r1.text[:150]}...' if len(r1.text) > 150 else f'Turn 1: {r1.text}')

# Turn 2: follow-up cost (uses context from turn 1)
r2 = chat.send_message('How much would priority processing cost for those?')
print(f'\nTurn 2: {r2.text[:150]}...' if len(r2.text) > 150 else f'\nTurn 2: {r2.text}')

# Turn 3: different tool
r3 = chat.send_message('Show me this month\'s query stats')
print(f'\nTurn 3: {r3.text[:150]}...' if len(r3.text) > 150 else f'\nTurn 3: {r3.text}')

# View history
print(f'\nTotal history messages: {len(chat.get_history())}')


## Cell 6: FunctionRegistry with Timeouts


In [ ]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
import logging

class FunctionRegistry:
    BLOCKED = {'delete_document', 'send_email', 'modify_access'}

    def __init__(self):
        self._funcs = {}
        self._timeouts = {}

    def register(self, name, func, timeout=30):
        self._funcs[name] = func
        self._timeouts[name] = timeout

    def execute(self, name, args):
        if name in self.BLOCKED:
            return {'error': f'{name} requires manual approval'}
        if name not in self._funcs:
            return {'error': f'Unknown function: {name}'}
        try:
            with ThreadPoolExecutor(max_workers=1) as pool:
                future = pool.submit(self._funcs[name], **args)
                result = future.result(timeout=self._timeouts[name])
            return {'result': result}
        except FuturesTimeout:
            return {'error': f'{name} timed out after {self._timeouts[name]}s'}
        except Exception as e:
            return {'error': f'Failed: {str(e)}'}

# Test
reg = FunctionRegistry()
reg.register('search_documents', search_documents, timeout=30)
reg.register('calculate_processing_cost', calculate_processing_cost, timeout=10)
reg.register('get_usage_stats', get_usage_stats, timeout=60)

print('Registry tests:')
print(f"  search: {reg.execute('search_documents', {'query': 'test'})}")
print(f"  blocked: {reg.execute('delete_document', {'id': 'D-01'})}")
print(f"  unknown: {reg.execute('nonexistent', {})}")
print(f"  bad args: {reg.execute('calculate_processing_cost', {'invalid': True})}")


## Cell 7: DocuMindAgent Class


In [ ]:
class DocuMindAgent:
    SYSTEM_PROMPT = (
        'You are DocuMind AI, a document intelligence assistant. '
        'Today is 2026-04-15. Use tools for document questions. '
        'When estimating costs, first search for real document counts. '
        'If search returns no results, explain clearly.')

    def __init__(self, client):
        self.client = client
        self.tools = [search_documents, calculate_processing_cost,
                      get_usage_stats]
        self.functions = FUNCTIONS

    def ask(self, question):
        return run_function_loop(
            client=self.client, prompt=question,
            tools=self.tools, functions=self.functions,
            system_instruction=self.SYSTEM_PROMPT, max_turns=8)

    def create_chat(self):
        return self.client.chats.create(
            model='gemini-3.6-flash',
            config=types.GenerateContentConfig(
                tools=self.tools,
                system_instruction=self.SYSTEM_PROMPT))

# Test single-turn
agent = DocuMindAgent(client)
print('=== Single-turn ===')
print(agent.ask('Find all legal documents and estimate standard processing cost'))

# Test multi-turn
print('\n=== Multi-turn ===')
chat = agent.create_chat()
print(chat.send_message('What invoices do we have?').text)
print(chat.send_message('How much to process them in bulk?').text)


## ✅ Lesson 6.2 Complete!

- ✅ While-loop dispatcher (loop until text)
- ✅ Conversation history with thought_signature preservation
- ✅ Sequential chaining (search → calculate using results)
- ✅ Parallel execution with id-based result mapping
- ✅ FunctionRegistry with timeouts and destructive-op blocking
- ✅ Error-as-data pattern (model explains failures naturally)
- ✅ Chat sessions with auto function calling
- ✅ Google Search + custom tools combined
- ✅ DocuMindAgent class (single-turn + multi-turn)

**Next: Lesson 6.3 — Google Search Grounding & Built-in Tools**
